# Paraphrase Dataset Generation

Creates balanced training/test splits from the original datasets,
then uses an OpenAI LLM to generate 4 paraphrases per statement (5 versions total
including the original)

## Base data
| File | Source | Rows | Composition |
|------|--------|------|-------------|
| train_cities.csv | cities.csv | 600 | 300 true + 300 false (city pairs preserved) |
| train_facts.csv | facts_true_false.csv | 600 | 300 true + 300 false |
| train_fever.csv | fever_small.csv | 600 | 300 true + 300 false |
| test_fever.csv | fever_small.csv | 2000 | 1000 true + 1000 false (no overlap with train) |

Selected 600 as facts_true_false has 613 statements (limiting size). 

## After paraphrasing
| File | Rows | = base × 5 |
|------|------|------------|
| para_train_cities.csv | 3,000 | 600 × 5 |
| para_train_facts.csv | 3,000 | 600 × 5 |
| para_train_fever.csv | 3,000 | 600 × 5 |
| para_test_fever.csv | 10,000 | 2,000 × 5 |

In [ ]:
# Imports
%pip install openai -q

import os
import re
import time
import random

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

os.makedirs('para_datasets', exist_ok=True)

os.environ["OPENAI_API_KEY"] = "HIDDEN"
client = OpenAI()  # reads OPENAI_API_KEY from environment
print("OpenAI client ready.")

---
## Create balanced base datasets

In [ ]:
# cities: select 300 city pairs (preserving true/false pairing)

cities_df = pd.read_csv('../datasets/cities.csv')
unique_cities = cities_df['city'].unique()
selected_cities = np.random.choice(unique_cities, size=300, replace=False)
train_cities = cities_df[cities_df['city'].isin(selected_cities)].copy()
train_cities = train_cities[['statement', 'label']].reset_index(drop=True)
train_cities.to_csv('para_datasets/train_cities.csv', index=False)

print(f"train_cities.csv: {len(train_cities)} rows  "
      f"(true: {(train_cities['label']==1).sum()}, false: {(train_cities['label']==0).sum()})")

# facts_true_false: 300 true + 300 false
facts_df = pd.read_csv('../datasets/facts_true_false.csv')
true_facts  = facts_df[facts_df['label'] == 1].sample(300, random_state=SEED)
false_facts = facts_df[facts_df['label'] == 0].sample(300, random_state=SEED)
train_facts = pd.concat([true_facts, false_facts]).sample(frac=1, random_state=SEED)
train_facts = train_facts[['statement', 'label']].reset_index(drop=True)
train_facts.to_csv('para_datasets/train_facts.csv', index=False)

print(f"train_facts.csv:  {len(train_facts)} rows  "
      f"(true: {(train_facts['label']==1).sum()}, false: {(train_facts['label']==0).sum()})")

# FEVER: train (300+300) + test (1000+1000), non-overlapping
fever_df = pd.read_csv('../datasets/fever_small.csv')
true_fever  = fever_df[fever_df['label'] == 1]
false_fever = fever_df[fever_df['label'] == 0]

train_true_f  = true_fever.sample(300, random_state=SEED)
train_false_f = false_fever.sample(300, random_state=SEED)
train_fever = pd.concat([train_true_f, train_false_f]).sample(frac=1, random_state=SEED)
train_fever = train_fever[['statement', 'label']].reset_index(drop=True)
train_fever.to_csv('para_datasets/train_fever.csv', index=False)

remaining_true  = true_fever.drop(train_true_f.index)
remaining_false = false_fever.drop(train_false_f.index)
test_true_f  = remaining_true.sample(1000, random_state=SEED)
test_false_f = remaining_false.sample(1000, random_state=SEED)
test_fever = pd.concat([test_true_f, test_false_f]).sample(frac=1, random_state=SEED)
test_fever = test_fever[['statement', 'label']].reset_index(drop=True)
test_fever.to_csv('para_datasets/test_fever.csv', index=False)

print(f"train_fever.csv:  {len(train_fever)} rows  "
      f"(true: {(train_fever['label']==1).sum()}, false: {(train_fever['label']==0).sum()})")
print(f"test_fever.csv:   {len(test_fever)} rows  "
      f"(true: {(test_fever['label']==1).sum()}, false: {(test_fever['label']==0).sum()})")

overlap = set(train_fever['statement']) & set(test_fever['statement'])
print(f"\nFEVER overlap check: {len(overlap)} shared statements (should be 0)")

---
## Paraphrase generation with ChatGPT

In [ ]:
N_PARAPHRASES = 4
MODEL = "gpt-4o-mini"

PARAPHRASE_PROMPT = """You are rewriting factual claims into alternative phrasings.

Given a statement, produce {n} different paraphrases that preserve the EXACT same claim.
Do not change any entities, numbers, relationships, or truth value.
If the original statement is incorrect, the paraphrases must preserve the same incorrect claim.
Do NOT correct errors or replace entities.

Requirements:
- Keep the meaning exactly the same.
- Preserve all named entities and numbers.
- Vary the sentence structure and wording.
- Each paraphrase should be a single sentence.
- Do not add new information.

Return the paraphrases as a numbered list.

Statement:
\"{statement}\""""


def generate_paraphrases(statement, n=N_PARAPHRASES, max_retries=3):
    """Call OpenAI to produce n paraphrases.  Returns list[str] of length n."""
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user",
                           "content": PARAPHRASE_PROMPT.format(n=n, statement=statement)}],
                temperature=0.7,
                max_tokens=600,
            )
            text = resp.choices[0].message.content.strip()
            paras = []
            for line in text.split('\n'):
                line = line.strip()
                m = re.match(r'^\d+[\.)\-]\s*', line)
                if m:
                    paras.append(line[m.end():].strip())
            if len(paras) >= n:
                return paras[:n]
            if attempt < max_retries - 1:
                time.sleep(1)
                continue
            # Pad with original on final attempt
            while len(paras) < n:
                paras.append(statement)
            return paras[:n]
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** (attempt + 1))
            else:
                print(f"  FAILED ({e}) — padding with original")
                return [statement] * n


def paraphrase_dataset(input_path, output_path, n=N_PARAPHRASES, delay=0.05):
    """
    Read base CSV, generate n paraphrases per statement, save expanded CSV.
    Columns: statement, label, original_statement, para_idx
    (para_idx 0 = original, 1..n = paraphrases)
    Supports resuming: skips rows already written.
    """
    df = pd.read_csv(input_path)
    expected_rows = len(df) * (1 + n)

    # Resume support: load existing rows
    start_from = 0
    existing_rows = []
    if os.path.exists(output_path):
        existing = pd.read_csv(output_path)
        if len(existing) >= expected_rows:
            print(f"  {output_path} already complete ({len(existing)} rows). Skipping.")
            return existing
        existing_rows = existing.to_dict('records')
        start_from = len(existing) // (1 + n)
        print(f"  Resuming from statement {start_from}/{len(df)} ({len(existing_rows)} rows on disk)")

    rows = list(existing_rows)
    save_every = 50  # checkpoint interval

    for i, row in tqdm(enumerate(df.itertuples()), total=len(df),
                       desc=os.path.basename(output_path), initial=start_from):
        if i < start_from:
            continue

        stmt, label = row.statement, row.label

        # Original
        rows.append({'statement': stmt, 'label': label,
                     'original_statement': stmt, 'para_idx': 0})

        # Paraphrases
        paras = generate_paraphrases(stmt, n=n)
        for j, p in enumerate(paras, 1):
            rows.append({'statement': p, 'label': label,
                         'original_statement': stmt, 'para_idx': j})

        # Periodic checkpoint
        if (i + 1) % save_every == 0:
            pd.DataFrame(rows).to_csv(output_path, index=False)

        time.sleep(delay)

    out_df = pd.DataFrame(rows)
    out_df.to_csv(output_path, index=False)
    print(f"  Saved {output_path}: {len(out_df)} rows")
    return out_df


print(f"Paraphrase config: model={MODEL}, N={N_PARAPHRASES}, prompt length ~{len(PARAPHRASE_PROMPT)} chars")

### Generate para_train_cities.csv

In [ ]:
para_cities = paraphrase_dataset(
    'para_datasets/train_cities.csv',
    'para_datasets/para_train_cities.csv',
)

### Generate para_train_facts.csv

In [ ]:
para_facts = paraphrase_dataset(
    'para_datasets/train_facts.csv',
    'para_datasets/para_train_facts.csv',
)

### Generate para_train_fever.csv

In [ ]:
para_train_fever = paraphrase_dataset(
    'para_datasets/train_fever.csv',
    'para_datasets/para_train_fever.csv',
)

### Generate para_test_fever.csv

In [ ]:
para_test_fever = paraphrase_dataset(
    'para_datasets/test_fever.csv',
    'para_datasets/para_test_fever.csv',
)